In [1]:
import pandas as pd

orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")
customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")
sellers = pd.read_csv("../data/raw/olist_sellers_dataset.csv")
order_items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")

In [2]:
customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 3.8 MB


In [3]:
(customers["customer_state"].value_counts(normalize=True) * 100).round(2).head(5)

customer_state
SP    41.98
RJ    12.92
MG    11.70
RS     5.50
PR     5.07
Name: proportion, dtype: float64

In [4]:
(sellers["seller_state"].value_counts(normalize=True) * 100).round(2).head(5)

seller_state
SP    59.74
PR    11.28
MG     7.88
SC     6.14
RJ     5.53
Name: proportion, dtype: float64

In [5]:
geo_orders = order_items.merge(
    orders[["order_id", "customer_id"]],
    on = "order_id",
    how="left"
).merge(
    customers[["customer_id", "customer_state"]],
    on = "customer_id",
    how="left"
).merge(
    sellers[["seller_id", "seller_state"]],
    on = "seller_id",
    how="left"
)

len(geo_orders)

112650

In [6]:
geo_orders["is_same_state"] = geo_orders["customer_state"] == geo_orders["seller_state"]

(geo_orders["is_same_state"].value_counts(normalize=True) * 100).round(2)

is_same_state
False    63.82
True     36.18
Name: proportion, dtype: float64

In [7]:
#We ensure the format is correct for dates in orders
date_cols = ["order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date"]
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

#Filtering only delivered with delivery date
delivered_orders = orders[
    (orders["order_status"] == "delivered") & (orders["order_delivered_customer_date"].notna())
].copy()

#We calculate reals days that have elapsed from the sale to delivery 
delivered_orders["delivery_days"] = (
    delivered_orders["order_delivered_customer_date"] - delivered_orders["order_purchase_timestamp"]
).dt.days

delivered_geo = delivered_orders.merge(
    customers[["customer_id", "customer_state"]],
    on="customer_id",
    how="left"
)



In [8]:
delivery_by_state = delivered_geo.groupby("customer_state")["delivery_days"].median().sort_values()

delivery_by_state.head(3)


customer_state
SP     7.0
MG    10.0
PR    10.0
Name: delivery_days, dtype: float64

In [9]:
delivery_by_state.tail(3)

customer_state
AP    24.0
AM    25.0
RR    25.0
Name: delivery_days, dtype: float64

In [12]:
delivered_geo["order_estimated_delivery_date"] = pd.to_datetime(delivered_geo["order_estimated_delivery_date"])

#We create the late flag: True if arrived after estimated delivery date
delivered_geo["is_late"] = delivered_geo["order_delivered_customer_date"] > delivered_geo["order_estimated_delivery_date"]

In [13]:
late_by_state = (delivered_geo.groupby("customer_state")["is_late"].mean() * 100).round(2).sort_values(ascending=False)

late_by_state.head(5)

customer_state
AL    23.93
MA    19.67
PI    15.97
CE    15.32
SE    15.22
Name: is_late, dtype: float64